<a href="https://colab.research.google.com/github/tini030205-debug/Deep-Learning-1Team/blob/main/food_classification_resnet50.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Colab 한글 폰트 설치
!apt-get update -qq
!apt-get install -y fonts-nanum -qq

print("나눔 폰트 설치 완료")

In [ ]:
# 기본 라이브러리
import os
import random
import zipfile
from pathlib import Path

# 데이터 확인 및 시각화
import matplotlib.pyplot as plt
from PIL import Image

# 딥러닝 관련 라이브러리
import numpy as np
import torch

In [ ]:
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Random Seed 고정
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# GPU 사용 시에도 seed 고정
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

# 재현성 향상을 위한 설정
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"Random Seed 고정 완료: {SEED}")

In [ ]:
# GPU 사용 가능 여부 확인
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("현재 사용 장치:", device)

if torch.cuda.is_available():
    print("GPU 이름:", torch.cuda.get_device_name(0))
else:
    print("GPU를 사용할 수 없습니다. Colab 런타임 유형에서 GPU를 선택하세요.")

In [ ]:
# zip 파일 경로 설정
zip_path = "/content/drive/MyDrive/딥러닝 데이터(300).zip"

# 압축 해제할 경로 설정
extract_dir = "/content/deep_learning_data"

# 압축 해제 경로 생성
os.makedirs(extract_dir, exist_ok=True)

# zip 파일 압축 해제
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print("압축 해제 완료")
print("압축 해제 위치:", extract_dir)

In [ ]:
# 압축 해제 후 최상위 폴더 확인
print("압축 해제 폴더 내부 목록:")
for item in os.listdir(extract_dir):
    print("-", item)

In [ ]:
# 압축 해제된 실제 데이터 폴더 경로 설정
data_root = os.path.join(extract_dir, "딥러닝 데이터(300)")

# Train / Validation / Test 경로 설정
train_dir = os.path.join(data_root, "Train")
valid_dir = os.path.join(data_root, "Validation")
test_dir = os.path.join(data_root, "Test")

# 경로 확인
print("데이터 루트 경로:", data_root)
print("Train 경로:", train_dir)
print("Validation 경로:", valid_dir)
print("Test 경로:", test_dir)

# 폴더 존재 여부 확인
print("\n폴더 존재 여부 확인")
print("Train 존재:", os.path.exists(train_dir))
print("Validation 존재:", os.path.exists(valid_dir))
print("Test 존재:", os.path.exists(test_dir))

In [ ]:
# Train 폴더 안의 하위 폴더명을 기준으로 클래스 자동 인식
class_names = sorted([
    folder for folder in os.listdir(train_dir)
    if os.path.isdir(os.path.join(train_dir, folder))
])

print("자동 인식된 클래스 이름:")
for idx, class_name in enumerate(class_names):
    print(f"{idx}: {class_name}")

In [ ]:
# 클래스 개수 출력
num_classes = len(class_names)

print("클래스 개수:", num_classes)

In [ ]:
# 이미지 확장자 목록
image_extensions = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]

# 클래스별 이미지 수를 세는 함수
def count_images_by_class(base_dir, class_names):
    class_counts = {}

    for class_name in class_names:
        class_dir = os.path.join(base_dir, class_name)

        image_count = len([
            file for file in os.listdir(class_dir)
            if os.path.isfile(os.path.join(class_dir, file))
            and Path(file).suffix.lower() in image_extensions
        ])

        class_counts[class_name] = image_count

    return class_counts

# Train / Validation / Test 각각 클래스별 이미지 수 확인
train_counts = count_images_by_class(train_dir, class_names)
valid_counts = count_images_by_class(valid_dir, class_names)
test_counts = count_images_by_class(test_dir, class_names)

print("[Train 클래스별 이미지 수]")
for class_name, count in train_counts.items():
    print(f"{class_name}: {count}장")

print("\n[Validation 클래스별 이미지 수]")
for class_name, count in valid_counts.items():
    print(f"{class_name}: {count}장")

print("\n[Test 클래스별 이미지 수]")
for class_name, count in test_counts.items():
    print(f"{class_name}: {count}장")

In [ ]:
# 전체 이미지 수 계산
total_train = sum(train_counts.values())
total_valid = sum(valid_counts.values())
total_test = sum(test_counts.values())
total_images = total_train + total_valid + total_test

print("Train 전체 이미지 수:", total_train)
print("Validation 전체 이미지 수:", total_valid)
print("Test 전체 이미지 수:", total_test)
print("전체 이미지 수:", total_images)

In [ ]:
# 클래스별 샘플 이미지 시각화
plt.figure(figsize=(20, 10))

for idx, class_name in enumerate(class_names):
    class_dir = os.path.join(train_dir, class_name)

    image_files = [
        file for file in os.listdir(class_dir)
        if os.path.isfile(os.path.join(class_dir, file))
        and Path(file).suffix.lower() in image_extensions
    ]

    # 클래스별 이미지 중 하나를 랜덤 선택
    sample_image_name = random.choice(image_files)
    sample_image_path = os.path.join(class_dir, sample_image_name)

    # 이미지 열기
    image = Image.open(sample_image_path).convert("RGB")

    # 시각화
    plt.subplot(2, 5, idx + 1)
    plt.imshow(image)
    plt.title(class_name, fontsize=14)
    plt.axis("off")

plt.suptitle("클래스별 샘플 이미지", fontsize=20)
plt.tight_layout()
plt.show()

In [ ]:
# PyTorch 데이터셋 및 데이터로더 관련 라이브러리
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# 경로 확인용
import os

In [ ]:
# 입력 이미지 크기 설정
IMG_SIZE = 224

# 배치 크기 설정
BATCH_SIZE = 32

# GPU 사용 여부에 따른 device 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("입력 이미지 크기:", IMG_SIZE)
print("배치 크기:", BATCH_SIZE)
print("사용 장치:", device)

In [ ]:
# ImageNet 사전학습 모델에서 일반적으로 사용하는 Normalize 값
# 추후 전이학습 모델을 사용할 때도 동일하게 사용할 수 있음
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

In [ ]:
# Train 데이터 Transform
# Train 데이터에는 데이터 증강을 적용함
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),          # 이미지를 224x224 크기로 변경
    transforms.RandomHorizontalFlip(p=0.5),           # 50% 확률로 좌우 반전
    transforms.RandomRotation(degrees=15),            # -15도 ~ +15도 범위에서 회전
    transforms.ColorJitter(                           # 밝기, 대비, 채도, 색조 변화
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.05
    ),
    transforms.ToTensor(),                            # 이미지를 Tensor 형태로 변환
    transforms.Normalize(                             # ImageNet 기준 정규화
        mean=imagenet_mean,
        std=imagenet_std
    )
])

# Validation / Test 데이터 Transform
# 검증과 테스트 데이터에는 증강을 적용하지 않음
valid_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),          # 이미지를 224x224 크기로 변경
    transforms.ToTensor(),                            # 이미지를 Tensor 형태로 변환
    transforms.Normalize(                             # ImageNet 기준 정규화
        mean=imagenet_mean,
        std=imagenet_std
    )
])

In [ ]:
# 1단계에서 설정한 경로가 정상인지 확인
print("Train 경로:", train_dir)
print("Validation 경로:", valid_dir)
print("Test 경로:", test_dir)

print("\n경로 존재 여부")
print("Train:", os.path.exists(train_dir))
print("Validation:", os.path.exists(valid_dir))
print("Test:", os.path.exists(test_dir))

In [ ]:
# ImageFolder는 폴더명을 클래스 이름으로 자동 인식함
# 한글 클래스명과 공백이 포함된 폴더명도 정상 처리 가능함
train_dataset = datasets.ImageFolder(
    root=train_dir,
    transform=train_transform
)

valid_dataset = datasets.ImageFolder(
    root=valid_dir,
    transform=valid_test_transform
)

test_dataset = datasets.ImageFolder(
    root=test_dir,
    transform=valid_test_transform
)

print("ImageFolder Dataset 생성 완료")

In [ ]:
# ImageFolder가 인식한 클래스 이름 확인
# folder 이름 기준으로 자동 정렬되어 라벨 번호가 부여됨
print("Train Dataset 클래스 목록:")
print(train_dataset.classes)

print("\n클래스 개수:", len(train_dataset.classes))

print("\n클래스명 → 라벨 인덱스 매핑:")
for class_name, label_idx in train_dataset.class_to_idx.items():
    print(f"{class_name}: {label_idx}")

In [ ]:
# Train / Validation / Test의 클래스 순서가 같은지 확인
print("Train 클래스:", train_dataset.classes)
print("Validation 클래스:", valid_dataset.classes)
print("Test 클래스:", test_dataset.classes)

print("\n클래스 목록 일치 여부")
print("Train == Validation:", train_dataset.classes == valid_dataset.classes)
print("Train == Test:", train_dataset.classes == test_dataset.classes)

In [ ]:
# 각 Dataset에 포함된 전체 이미지 수 확인
print("Train Dataset 이미지 수:", len(train_dataset))
print("Validation Dataset 이미지 수:", len(valid_dataset))
print("Test Dataset 이미지 수:", len(test_dataset))

print("\n전체 이미지 수:", len(train_dataset) + len(valid_dataset) + len(test_dataset))

In [ ]:
# Colab 환경에서 일반적으로 사용 가능한 num_workers 설정
# 오류가 발생하면 num_workers=0으로 바꾸면 됨
NUM_WORKERS = 2

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,              # Train은 데이터 순서를 섞음
    num_workers=NUM_WORKERS,
    pin_memory=True
)

valid_loader = DataLoader(
    dataset=valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,             # Validation은 데이터 순서를 섞지 않음
    num_workers=NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,             # Test는 데이터 순서를 섞지 않음
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print("DataLoader 생성 완료")

In [ ]:
# Train DataLoader에서 배치 하나 꺼내기
images, labels = next(iter(train_loader))

print("이미지 batch shape:", images.shape)
print("라벨 batch shape:", labels.shape)

print("\n이미지 Tensor 자료형:", images.dtype)
print("라벨 Tensor 자료형:", labels.dtype)

print("\n이미지 batch 최소값:", images.min().item())
print("이미지 batch 최대값:", images.max().item())

print("\n라벨 예시:", labels[:10])

In [ ]:
# Validation DataLoader에서 배치 하나 확인
valid_images, valid_labels = next(iter(valid_loader))

print("[Validation Batch]")
print("이미지 shape:", valid_images.shape)
print("라벨 shape:", valid_labels.shape)

# Test DataLoader에서 배치 하나 확인
test_images, test_labels = next(iter(test_loader))

print("\n[Test Batch]")
print("이미지 shape:", test_images.shape)
print("라벨 shape:", test_labels.shape)

In [ ]:
# =========================
# 3단계-1. 라이브러리 import
# =========================

import os
import random
import time
import copy

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import models
from torch.optim.lr_scheduler import CosineAnnealingLR

In [ ]:
# =========================
# 3단계-2. Random Seed 고정
# =========================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("Random Seed 고정 완료:", SEED)

In [ ]:
# =========================
# 3단계-3. Scratch ResNet50 모델 정의
# =========================

# 클래스 개수 확인
NUM_CLASSES = 10

# ImageNet 사전학습 가중치를 사용하지 않는 ResNet50 생성
# weights=None 이므로 Scratch 모델임
scratch_model = models.resnet50(weights=None)

# 기존 ResNet50의 마지막 FC Layer 입력 feature 수 확인
in_features = scratch_model.fc.in_features

# 마지막 FC Layer를 10개 클래스 출력으로 변경
scratch_model.fc = nn.Linear(in_features, NUM_CLASSES)

# 모델을 GPU 또는 CPU로 이동
scratch_model = scratch_model.to(device)

print("Scratch ResNet50 모델 생성 완료")
print("마지막 FC Layer:", scratch_model.fc)
print("출력 클래스 수:", NUM_CLASSES)
print("사용 장치:", device)

In [ ]:
# =========================
# 3단계-4. Loss / Optimizer / Scheduler 정의
# =========================

# Loss Function
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = optim.Adam(
    scratch_model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

# Scheduler
scheduler = CosineAnnealingLR(
    optimizer,
    T_max=100
)

print("Loss Function:", criterion)
print("Optimizer:", optimizer)
print("Scheduler:", scheduler)

In [ ]:
# =========================
# 3단계-5. EarlyStopping 클래스 정의
# =========================

class EarlyStopping:
    """
    Validation Loss를 기준으로 Early Stopping을 수행하는 클래스
    patience 동안 val_loss가 개선되지 않으면 학습을 중단함
    """

    def __init__(self, patience=10, min_delta=0.0):
        """
        patience: 개선이 없어도 기다릴 epoch 수
        min_delta: 개선으로 인정할 최소 감소량
        """
        self.patience = patience
        self.min_delta = min_delta

        self.best_loss = np.inf
        self.counter = 0
        self.early_stop = False

    def __call__(self, val_loss):
        """
        현재 epoch의 validation loss를 입력받아 early stopping 여부 판단
        """

        # val_loss가 best_loss보다 충분히 감소한 경우
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0

        # 개선되지 않은 경우
        else:
            self.counter += 1
            print(f"EarlyStopping Counter: {self.counter}/{self.patience}")

            if self.counter >= self.patience:
                self.early_stop = True

In [ ]:
# =========================
# 3단계-6. 정확도 계산 함수 정의
# =========================

def calculate_accuracy(outputs, labels):
    """
    모델 출력값과 실제 라벨을 이용해 accuracy 계산
    """

    # outputs에서 가장 큰 값을 가진 클래스 index 선택
    _, preds = torch.max(outputs, dim=1)

    # 맞춘 개수 계산
    correct = torch.sum(preds == labels).item()

    return correct

In [ ]:
# =========================
# 3단계-7. train_model 함수 정의
# =========================

def train_model(
    model,
    train_loader,
    valid_loader,
    criterion,
    optimizer,
    scheduler,
    num_epochs=100,
    patience=10,
    save_path="scratch_best_model.pth",
    history_path="scratch_history.csv"
):
    """
    ResNet50 Scratch 모델 학습 함수

    저장 기준:
    - best model: validation loss가 가장 낮을 때 저장
    - history: epoch별 train_loss, val_loss, train_acc, val_acc, learning_rate 저장
    """

    since = time.time()

    # EarlyStopping 객체 생성
    early_stopping = EarlyStopping(patience=patience)

    # best val_loss 초기화
    best_val_loss = np.inf

    # best model weight 저장용
    best_model_wts = copy.deepcopy(model.state_dict())

    # 학습 기록 저장 리스트
    history = []

    print("학습 시작")
    print("=" * 80)

    for epoch in range(num_epochs):
        print(f"\nEpoch [{epoch + 1}/{num_epochs}]")
        print("-" * 80)

        # =========================
        # Train 단계
        # =========================
        model.train()

        train_loss_sum = 0.0
        train_correct = 0
        train_total = 0

        for images, labels in train_loader:
            # 데이터를 device로 이동
            images = images.to(device)
            labels = labels.to(device)

            # 이전 gradient 초기화
            optimizer.zero_grad()

            # Forward
            outputs = model(images)

            # Loss 계산
            loss = criterion(outputs, labels)

            # Backward
            loss.backward()

            # Optimizer update
            optimizer.step()

            # 통계 누적
            batch_size = images.size(0)
            train_loss_sum += loss.item() * batch_size
            train_correct += calculate_accuracy(outputs, labels)
            train_total += batch_size

        # epoch 단위 train loss / accuracy 계산
        train_loss = train_loss_sum / train_total
        train_acc = train_correct / train_total

        # =========================
        # Validation 단계
        # =========================
        model.eval()

        val_loss_sum = 0.0
        val_correct = 0
        val_total = 0

        # validation에서는 gradient 계산 비활성화
        with torch.no_grad():
            for images, labels in valid_loader:
                images = images.to(device)
                labels = labels.to(device)

                # Forward
                outputs = model(images)

                # Loss 계산
                loss = criterion(outputs, labels)

                # 통계 누적
                batch_size = images.size(0)
                val_loss_sum += loss.item() * batch_size
                val_correct += calculate_accuracy(outputs, labels)
                val_total += batch_size

        # epoch 단위 validation loss / accuracy 계산
        val_loss = val_loss_sum / val_total
        val_acc = val_correct / val_total

        # 현재 learning rate 확인
        current_lr = optimizer.param_groups[0]["lr"]

        # Scheduler update
        scheduler.step()

        # history 저장
        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "train_acc": train_acc,
            "val_acc": val_acc,
            "learning_rate": current_lr
        })

        # epoch 결과 출력
        print(
            f"train_loss: {train_loss:.4f} | "
            f"val_loss: {val_loss:.4f} | "
            f"train_acc: {train_acc:.4f} | "
            f"val_acc: {val_acc:.4f} | "
            f"learning_rate: {current_lr:.8f}"
        )

        # =========================
        # Best Model 저장 기준: val_loss
        # =========================
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_wts = copy.deepcopy(model.state_dict())

            torch.save(model.state_dict(), save_path)

            print(f"Best model 저장 완료: {save_path}")
            print(f"Best val_loss: {best_val_loss:.4f}")

        # =========================
        # EarlyStopping 검사
        # monitor = val_loss
        # =========================
        early_stopping(val_loss)

        if early_stopping.early_stop:
            print("\nEarlyStopping 발생")
            print(f"{patience} epoch 동안 val_loss 개선이 없어 학습을 중단합니다.")
            break

    # 전체 학습 시간 계산
    time_elapsed = time.time() - since

    print("\n" + "=" * 80)
    print("학습 종료")
    print(f"총 학습 시간: {time_elapsed // 60:.0f}분 {time_elapsed % 60:.0f}초")
    print(f"최고 val_loss: {best_val_loss:.4f}")

    # best model weight 로드
    model.load_state_dict(best_model_wts)

    # history를 CSV로 저장
    history_df = pd.DataFrame(history)
    history_df.to_csv(history_path, index=False, encoding="utf-8-sig")

    print(f"학습 기록 저장 완료: {history_path}")
    print(f"Best model 저장 파일: {save_path}")

    return model, history_df

In [ ]:
# =========================
# 3단계-8. Scratch ResNet50 학습 실행
# =========================

EPOCHS = 100
PATIENCE = 10

scratch_model, scratch_history = train_model(
    model=scratch_model,
    train_loader=train_loader,
    valid_loader=valid_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    num_epochs=EPOCHS,
    patience=PATIENCE,
    save_path="scratch_best_model.pth",
    history_path="scratch_history.csv"
)

In [ ]:
# =========================
# 3단계-9. 저장 파일 확인
# =========================

print("scratch_best_model.pth 존재 여부:", os.path.exists("scratch_best_model.pth"))
print("scratch_history.csv 존재 여부:", os.path.exists("scratch_history.csv"))

In [ ]:
# =========================
# 3단계-10. 학습 기록 확인
# =========================

scratch_history.head()
scratch_history.tail()

In [ ]:
# =========================
# 3단계-11. Loss 그래프 시각화
# =========================

plt.figure(figsize=(8, 5))
plt.plot(scratch_history["epoch"], scratch_history["train_loss"], label="Train Loss")
plt.plot(scratch_history["epoch"], scratch_history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Scratch ResNet50 Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# =========================
# 3단계-12. Accuracy 그래프 시각화
# =========================

plt.figure(figsize=(8, 5))
plt.plot(scratch_history["epoch"], scratch_history["train_acc"], label="Train Accuracy")
plt.plot(scratch_history["epoch"], scratch_history["val_acc"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Scratch ResNet50 Accuracy")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# =========================
# 4단계-1. 라이브러리 import
# =========================

import os
import time
import copy
import random

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import models
from torch.optim.lr_scheduler import CosineAnnealingLR

In [ ]:
# =========================
# 4단계-2. Random Seed 고정
# =========================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("Random Seed 고정 완료:", SEED)

In [ ]:
# =========================
# 4단계-3. ImageNet 사전학습 ResNet50 모델 정의
# =========================

NUM_CLASSES = 10

# torchvision 최신 weights 방식 사용
weights = models.ResNet50_Weights.DEFAULT

# ImageNet 사전학습 ResNet50 로드
transfer_model = models.resnet50(weights=weights)

# 기존 FC Layer 입력 feature 수 확인
in_features = transfer_model.fc.in_features

# 마지막 FC Layer를 10개 클래스 출력으로 변경
# FC Layer 앞에 Dropout(p=0.5) 적용
transfer_model.fc = nn.Sequential(
    nn.Dropout(p=0.5),
    nn.Linear(in_features, NUM_CLASSES)
)

# 모델을 GPU 또는 CPU로 이동
transfer_model = transfer_model.to(device)

print("Transfer Learning ResNet50 모델 생성 완료")
print("사용 weights:", weights)
print("마지막 FC Layer:")
print(transfer_model.fc)
print("출력 클래스 수:", NUM_CLASSES)
print("사용 장치:", device)

In [ ]:
# =========================
# 4단계-4. Feature Extraction 단계 설정
# backbone freeze, FC Layer만 학습
# =========================

# 전체 파라미터 freeze
for param in transfer_model.parameters():
    param.requires_grad = False

# 마지막 FC Layer만 학습 가능하도록 설정
for param in transfer_model.fc.parameters():
    param.requires_grad = True

# 학습 가능한 파라미터 수 확인
trainable_params = sum(p.numel() for p in transfer_model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in transfer_model.parameters())

print("Feature Extraction 설정 완료")
print(f"전체 파라미터 수: {total_params:,}")
print(f"학습 가능한 파라미터 수: {trainable_params:,}")

In [ ]:
# =========================
# 4단계-5. Transfer Learning 학습 함수 정의
# =========================

def train_transfer_phase(
    model,
    train_loader,
    valid_loader,
    criterion,
    optimizer,
    scheduler,
    phase_name,
    num_epochs,
    patience,
    best_val_loss,
    best_model_wts,
    history,
    save_path="transfer_best_model.pth"
):
    """
    Transfer Learning의 한 단계 학습 함수

    phase_name:
    - "feature_extraction"
    - "fine_tuning"

    best model 저장 기준:
    - 전체 phase를 통틀어 validation loss가 가장 낮을 때 저장
    """

    since = time.time()

    early_stopping = EarlyStopping(patience=patience)

    print(f"\n[{phase_name}] 학습 시작")
    print("=" * 80)

    for epoch in range(num_epochs):
        print(f"\nPhase: {phase_name} | Epoch [{epoch + 1}/{num_epochs}]")
        print("-" * 80)

        # =========================
        # Train 단계
        # =========================
        model.train()

        train_loss_sum = 0.0
        train_correct = 0
        train_total = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            batch_size = images.size(0)
            train_loss_sum += loss.item() * batch_size
            train_correct += calculate_accuracy(outputs, labels)
            train_total += batch_size

        train_loss = train_loss_sum / train_total
        train_acc = train_correct / train_total

        # =========================
        # Validation 단계
        # =========================
        model.eval()

        val_loss_sum = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for images, labels in valid_loader:
                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)

                batch_size = images.size(0)
                val_loss_sum += loss.item() * batch_size
                val_correct += calculate_accuracy(outputs, labels)
                val_total += batch_size

        val_loss = val_loss_sum / val_total
        val_acc = val_correct / val_total

        # 현재 learning rate 확인
        current_lr = optimizer.param_groups[0]["lr"]

        # Scheduler update
        scheduler.step()

        # history 저장
        history.append({
            "phase": phase_name,
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "train_acc": train_acc,
            "val_acc": val_acc,
            "learning_rate": current_lr
        })

        # epoch별 결과 출력
        print(
            f"train_loss: {train_loss:.4f} | "
            f"val_loss: {val_loss:.4f} | "
            f"train_acc: {train_acc:.4f} | "
            f"val_acc: {val_acc:.4f} | "
            f"learning_rate: {current_lr:.8f}"
        )

        # =========================
        # Best Model 저장 기준: val_loss
        # =========================
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_wts = copy.deepcopy(model.state_dict())

            torch.save(model.state_dict(), save_path)

            print(f"Best model 저장 완료: {save_path}")
            print(f"Best val_loss: {best_val_loss:.4f}")

        # =========================
        # EarlyStopping
        # monitor = val_loss
        # =========================
        early_stopping(val_loss)

        if early_stopping.early_stop:
            print(f"\n[{phase_name}] EarlyStopping 발생")
            print(f"{patience} epoch 동안 val_loss 개선이 없어 해당 phase 학습을 중단합니다.")
            break

    time_elapsed = time.time() - since

    print("\n" + "=" * 80)
    print(f"[{phase_name}] 학습 종료")
    print(f"소요 시간: {time_elapsed // 60:.0f}분 {time_elapsed % 60:.0f}초")
    print(f"현재까지 최고 val_loss: {best_val_loss:.4f}")

    return model, best_val_loss, best_model_wts, history

In [ ]:
# =========================
# 4단계-6. 공통 학습 설정
# =========================

# Epoch 수 설정
FEATURE_EPOCHS = 30
FINETUNE_EPOCHS = 100

# EarlyStopping patience
PATIENCE = 10

# Loss Function
criterion = nn.CrossEntropyLoss()

# best model 관리 변수
best_val_loss = np.inf
best_model_wts = copy.deepcopy(transfer_model.state_dict())

# history 저장 리스트
transfer_history = []

print("공통 학습 설정 완료")
print("Feature Extraction Epochs:", FEATURE_EPOCHS)
print("Fine-tuning Epochs:", FINETUNE_EPOCHS)
print("Patience:", PATIENCE)
print("Loss Function:", criterion)

In [ ]:
# =========================
# 4단계-7. Feature Extraction Optimizer / Scheduler
# =========================

# FC Layer만 학습
feature_optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, transfer_model.parameters()),
    lr=1e-3,
    weight_decay=1e-4
)

feature_scheduler = CosineAnnealingLR(
    feature_optimizer,
    T_max=FEATURE_EPOCHS
)

print("Feature Extraction Optimizer / Scheduler 설정 완료")
print("Learning Rate:", feature_optimizer.param_groups[0]["lr"])
print("Weight Decay:", feature_optimizer.param_groups[0]["weight_decay"])

In [ ]:
# =========================
# 4단계-8. Feature Extraction 학습 실행
# =========================

transfer_model, best_val_loss, best_model_wts, transfer_history = train_transfer_phase(
    model=transfer_model,
    train_loader=train_loader,
    valid_loader=valid_loader,
    criterion=criterion,
    optimizer=feature_optimizer,
    scheduler=feature_scheduler,
    phase_name="feature_extraction",
    num_epochs=FEATURE_EPOCHS,
    patience=PATIENCE,
    best_val_loss=best_val_loss,
    best_model_wts=best_model_wts,
    history=transfer_history,
    save_path="transfer_best_model.pth"
)

In [ ]:
# =========================
# 4단계-9. Fine-tuning 단계 설정
# 전체 layer unfreeze
# =========================

# 전체 layer 학습 가능하도록 변경
for param in transfer_model.parameters():
    param.requires_grad = True

# 학습 가능한 파라미터 수 확인
trainable_params = sum(p.numel() for p in transfer_model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in transfer_model.parameters())

print("Fine-tuning 설정 완료")
print(f"전체 파라미터 수: {total_params:,}")
print(f"학습 가능한 파라미터 수: {trainable_params:,}")

In [ ]:
# =========================
# 4단계-10. Fine-tuning Optimizer / Scheduler
# =========================

# 전체 네트워크 미세 조정
finetune_optimizer = optim.Adam(
    transfer_model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

finetune_scheduler = CosineAnnealingLR(
    finetune_optimizer,
    T_max=FINETUNE_EPOCHS
)

print("Fine-tuning Optimizer / Scheduler 설정 완료")
print("Learning Rate:", finetune_optimizer.param_groups[0]["lr"])
print("Weight Decay:", finetune_optimizer.param_groups[0]["weight_decay"])

In [ ]:
# =========================
# 4단계-11. Fine-tuning 학습 실행
# =========================

transfer_model, best_val_loss, best_model_wts, transfer_history = train_transfer_phase(
    model=transfer_model,
    train_loader=train_loader,
    valid_loader=valid_loader,
    criterion=criterion,
    optimizer=finetune_optimizer,
    scheduler=finetune_scheduler,
    phase_name="fine_tuning",
    num_epochs=FINETUNE_EPOCHS,
    patience=PATIENCE,
    best_val_loss=best_val_loss,
    best_model_wts=best_model_wts,
    history=transfer_history,
    save_path="transfer_best_model.pth"
)

In [ ]:
# =========================
# 4단계-12. Best Model 로드 및 transfer_history.csv 저장
# =========================

# 전체 phase 중 가장 val_loss가 낮았던 weight 로드
transfer_model.load_state_dict(best_model_wts)

# history DataFrame 생성
transfer_history_df = pd.DataFrame(transfer_history)

# CSV 저장
transfer_history_df.to_csv(
    "transfer_history.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Transfer Learning 학습 완료")
print(f"Best val_loss: {best_val_loss:.4f}")
print("Best model 저장 파일: transfer_best_model.pth")
print("History 저장 파일: transfer_history.csv")

In [ ]:
# =========================
# 4단계-13. 저장 파일 확인
# =========================

print("transfer_best_model.pth 존재 여부:", os.path.exists("transfer_best_model.pth"))
print("transfer_history.csv 존재 여부:", os.path.exists("transfer_history.csv"))

In [ ]:
# =========================
# 4단계-14. History 확인
# =========================

transfer_history_df.head()
transfer_history_df.tail()

In [ ]:
# =========================
# 4단계-15. Phase별 최고 성능 확인
# =========================

summary = transfer_history_df.groupby("phase").agg({
    "train_loss": "min",
    "val_loss": "min",
    "train_acc": "max",
    "val_acc": "max"
})

summary

In [ ]:
# =========================
# 4단계-16. Loss 그래프 시각화
# =========================

plt.figure(figsize=(10, 5))

plt.plot(
    range(1, len(transfer_history_df) + 1),
    transfer_history_df["train_loss"],
    label="Train Loss"
)

plt.plot(
    range(1, len(transfer_history_df) + 1),
    transfer_history_df["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Total Epoch")
plt.ylabel("Loss")
plt.title("Transfer Learning ResNet50 Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# =========================
# 4단계-17. Accuracy 그래프 시각화
# =========================

plt.figure(figsize=(10, 5))

plt.plot(
    range(1, len(transfer_history_df) + 1),
    transfer_history_df["train_acc"],
    label="Train Accuracy"
)

plt.plot(
    range(1, len(transfer_history_df) + 1),
    transfer_history_df["val_acc"],
    label="Validation Accuracy"
)

plt.xlabel("Total Epoch")
plt.ylabel("Accuracy")
plt.title("Transfer Learning ResNet50 Accuracy")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# =========================
# 5단계-1. 라이브러리 import
# =========================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torchvision import models

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

In [ ]:
# =========================
# 5단계-2. 결과 저장 폴더 생성
# =========================

RESULT_DIR = "results"
os.makedirs(RESULT_DIR, exist_ok=True)

print("결과 저장 폴더 생성 완료:", RESULT_DIR)

In [ ]:
# =========================
# 5단계-3. 클래스 이름 확인
# =========================

# ImageFolder 기준 클래스 이름 사용
# class_names 변수가 없다면 train_dataset.classes를 사용
try:
    class_names = class_names
except NameError:
    class_names = train_dataset.classes

NUM_CLASSES = len(class_names)

print("클래스 개수:", NUM_CLASSES)
print("클래스 목록:")
for idx, name in enumerate(class_names):
    print(f"{idx}: {name}")

In [ ]:
# =========================
# 5단계-4. Scratch ResNet50 모델 정의 및 best weight 로드
# =========================

# Scratch ResNet50 생성
scratch_eval_model = models.resnet50(weights=None)

# 마지막 FC Layer를 10개 클래스 출력으로 변경
in_features = scratch_eval_model.fc.in_features
scratch_eval_model.fc = nn.Linear(in_features, NUM_CLASSES)

# 저장된 best model weight 로드
scratch_eval_model.load_state_dict(
    torch.load("scratch_best_model.pth", map_location=device)
)

# device로 이동
scratch_eval_model = scratch_eval_model.to(device)

# 평가 모드
scratch_eval_model.eval()

print("Scratch best model 로드 완료")

In [ ]:
# =========================
# 5단계-5. Transfer Learning ResNet50 모델 정의 및 best weight 로드
# =========================

# 최신 torchvision weights 방식 사용
weights = models.ResNet50_Weights.DEFAULT

# ImageNet 사전학습 ResNet50 생성
transfer_eval_model = models.resnet50(weights=weights)

# 마지막 FC Layer를 Dropout + Linear 구조로 변경
in_features = transfer_eval_model.fc.in_features
transfer_eval_model.fc = nn.Sequential(
    nn.Dropout(p=0.5),
    nn.Linear(in_features, NUM_CLASSES)
)

# 저장된 best model weight 로드
transfer_eval_model.load_state_dict(
    torch.load("transfer_best_model.pth", map_location=device)
)

# device로 이동
transfer_eval_model = transfer_eval_model.to(device)

# 평가 모드
transfer_eval_model.eval()

print("Transfer Learning best model 로드 완료")

In [ ]:
# =========================
# 5단계-6. Test 평가 함수 정의
# =========================

def evaluate_model(model, data_loader, device):
    """
    Test 데이터셋에 대해 모델을 평가하는 함수

    반환:
    - y_true: 실제 라벨
    - y_pred: 예측 라벨
    - test_accuracy: Test Accuracy
    """

    model.eval()

    y_true = []
    y_pred = []

    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, preds = torch.max(outputs, dim=1)

            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    test_accuracy = accuracy_score(y_true, y_pred)

    return np.array(y_true), np.array(y_pred), test_accuracy

In [ ]:
# =========================
# 5단계-7. Scratch 모델 Test 평가
# =========================

scratch_y_true, scratch_y_pred, scratch_test_acc = evaluate_model(
    model=scratch_eval_model,
    data_loader=test_loader,
    device=device
)

print("Scratch ResNet50 Test Accuracy:", round(scratch_test_acc, 4))

In [ ]:
# =========================
# 5단계-8. Transfer Learning 모델 Test 평가
# =========================

transfer_y_true, transfer_y_pred, transfer_test_acc = evaluate_model(
    model=transfer_eval_model,
    data_loader=test_loader,
    device=device
)

print("Transfer Learning ResNet50 Test Accuracy:", round(transfer_test_acc, 4))

In [ ]:
# =========================
# 5단계-9. Classification Report 출력
# =========================

print("=" * 80)
print("Scratch ResNet50 Classification Report")
print("=" * 80)

scratch_report_text = classification_report(
    scratch_y_true,
    scratch_y_pred,
    target_names=class_names,
    digits=4
)

print(scratch_report_text)


print("=" * 80)
print("Transfer Learning ResNet50 Classification Report")
print("=" * 80)

transfer_report_text = classification_report(
    transfer_y_true,
    transfer_y_pred,
    target_names=class_names,
    digits=4
)

print(transfer_report_text)

In [ ]:
# =========================
# 5단계-10. 클래스별 Precision, Recall, F1-score 출력
# =========================

scratch_report_dict = classification_report(
    scratch_y_true,
    scratch_y_pred,
    target_names=class_names,
    output_dict=True,
    digits=4
)

transfer_report_dict = classification_report(
    transfer_y_true,
    transfer_y_pred,
    target_names=class_names,
    output_dict=True,
    digits=4
)

scratch_class_metrics = pd.DataFrame(scratch_report_dict).transpose()
transfer_class_metrics = pd.DataFrame(transfer_report_dict).transpose()

print("[Scratch 모델 클래스별 성능]")
display(scratch_class_metrics)

print("\n[Transfer Learning 모델 클래스별 성능]")
display(transfer_class_metrics)

In [ ]:
# =========================
# 5단계-11. Confusion Matrix 시각화 및 저장 함수 정의
# =========================

def plot_and_save_confusion_matrix(y_true, y_pred, class_names, title, save_path):
    """
    Confusion Matrix를 시각화하고 이미지 파일로 저장하는 함수
    """

    cm = confusion_matrix(y_true, y_pred)

    fig, ax = plt.subplots(figsize=(12, 10))

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=class_names
    )

    disp.plot(
        ax=ax,
        cmap="Blues",
        xticks_rotation=45,
        values_format="d"
    )

    plt.title(title, fontsize=16)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Confusion Matrix 저장 완료:", save_path)

In [ ]:
# =========================
# 5단계-12. Scratch Confusion Matrix 저장
# =========================

scratch_cm_path = os.path.join(RESULT_DIR, "scratch_confusion_matrix.png")

plot_and_save_confusion_matrix(
    y_true=scratch_y_true,
    y_pred=scratch_y_pred,
    class_names=class_names,
    title="Scratch ResNet50 Confusion Matrix",
    save_path=scratch_cm_path
)

In [ ]:
# =========================
# 5단계-13. Transfer Learning Confusion Matrix 저장
# =========================

transfer_cm_path = os.path.join(RESULT_DIR, "transfer_confusion_matrix.png")

plot_and_save_confusion_matrix(
    y_true=transfer_y_true,
    y_pred=transfer_y_pred,
    class_names=class_names,
    title="Transfer Learning ResNet50 Confusion Matrix",
    save_path=transfer_cm_path
)

In [ ]:
# =========================
# 5단계-14. Scratch vs Transfer Learning 성능 비교 표 생성
# =========================

scratch_macro_precision = scratch_report_dict["macro avg"]["precision"]
scratch_macro_recall = scratch_report_dict["macro avg"]["recall"]
scratch_macro_f1 = scratch_report_dict["macro avg"]["f1-score"]

transfer_macro_precision = transfer_report_dict["macro avg"]["precision"]
transfer_macro_recall = transfer_report_dict["macro avg"]["recall"]
transfer_macro_f1 = transfer_report_dict["macro avg"]["f1-score"]

comparison_summary = pd.DataFrame({
    "model": ["Scratch ResNet50", "Transfer Learning ResNet50"],
    "test_accuracy": [scratch_test_acc, transfer_test_acc],
    "macro_precision": [scratch_macro_precision, transfer_macro_precision],
    "macro_recall": [scratch_macro_recall, transfer_macro_recall],
    "macro_f1_score": [scratch_macro_f1, transfer_macro_f1]
})

comparison_summary

In [ ]:
# =========================
# 5단계-15. 성능 비교 표 CSV 저장
# =========================

summary_csv_path = os.path.join(RESULT_DIR, "model_comparison_summary.csv")

comparison_summary.to_csv(
    summary_csv_path,
    index=False,
    encoding="utf-8-sig"
)

print("성능 비교 요약 CSV 저장 완료:", summary_csv_path)

In [ ]:
# =========================
# 5단계-16. Scratch / Transfer history 불러오기
# =========================

scratch_history = pd.read_csv("scratch_history.csv")
transfer_history = pd.read_csv("transfer_history.csv")

print("Scratch history shape:", scratch_history.shape)
print("Transfer history shape:", transfer_history.shape)

display(scratch_history.head())
display(transfer_history.head())

In [ ]:
# =========================
# 5단계-17. Transfer history 전체 epoch 번호 생성
# =========================

# transfer_history는 feature_extraction과 fine_tuning epoch가 각각 1부터 시작할 수 있으므로
# 그래프 비교를 위해 전체 순서 기준 total_epoch를 추가함
transfer_history["total_epoch"] = range(1, len(transfer_history) + 1)

# scratch_history도 비교를 위해 total_epoch 생성
scratch_history["total_epoch"] = range(1, len(scratch_history) + 1)

display(transfer_history.tail())

In [ ]:
# =========================
# 5단계-18. Validation Accuracy / Validation Loss 비교 그래프 저장
# =========================

comparison_curve_path = os.path.join(RESULT_DIR, "model_comparison_curve.png")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# -------------------------
# Validation Loss 비교
# -------------------------
axes[0].plot(
    scratch_history["total_epoch"],
    scratch_history["val_loss"],
    label="Scratch Val Loss"
)

axes[0].plot(
    transfer_history["total_epoch"],
    transfer_history["val_loss"],
    label="Transfer Val Loss"
)

axes[0].set_title("Validation Loss Comparison")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Validation Loss")
axes[0].legend()
axes[0].grid(True)

# -------------------------
# Validation Accuracy 비교
# -------------------------
axes[1].plot(
    scratch_history["total_epoch"],
    scratch_history["val_acc"],
    label="Scratch Val Accuracy"
)

axes[1].plot(
    transfer_history["total_epoch"],
    transfer_history["val_acc"],
    label="Transfer Val Accuracy"
)

axes[1].set_title("Validation Accuracy Comparison")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Validation Accuracy")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig(comparison_curve_path, dpi=300, bbox_inches="tight")
plt.show()

print("모델 비교 그래프 저장 완료:", comparison_curve_path)

In [ ]:
# =========================
# 5단계-19. 저장 결과 최종 확인
# =========================

expected_files = [
    "scratch_confusion_matrix.png",
    "transfer_confusion_matrix.png",
    "model_comparison_curve.png",
    "model_comparison_summary.csv"
]

print("results 폴더 저장 파일 확인")
print("=" * 50)

for file_name in expected_files:
    file_path = os.path.join(RESULT_DIR, file_name)
    print(f"{file_name}: {os.path.exists(file_path)}")

In [ ]:
# =========================
# 6단계-1. 라이브러리 import
# =========================

import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn.functional as F

In [ ]:
# =========================
# 6단계-2. 결과 저장 폴더 생성
# =========================

RESULT_DIR = "results"
os.makedirs(RESULT_DIR, exist_ok=True)

print("결과 저장 폴더:", RESULT_DIR)

In [ ]:
# =========================
# 6단계-3. 오분류 분석 대상 모델 선택
# =========================

# 기본 분석 대상: Transfer Learning 모델
analysis_model = transfer_eval_model
analysis_model_name = "Transfer Learning ResNet50"

analysis_model.eval()

print("오분류 분석 대상 모델:", analysis_model_name)

In [ ]:
# =========================
# 6단계-4. Test 데이터 전체 예측 결과 수집
# =========================

all_results = []

analysis_model.eval()

sample_index = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = analysis_model(images)

        # softmax를 이용해 클래스별 예측 확률 계산
        probs = F.softmax(outputs, dim=1)

        # 가장 확률이 높은 클래스 선택
        pred_probs, preds = torch.max(probs, dim=1)

        # batch 내부 데이터 하나씩 저장
        for i in range(images.size(0)):
            true_label_idx = labels[i].item()
            pred_label_idx = preds[i].item()
            pred_prob = pred_probs[i].item()

            # ImageFolder의 samples에는 이미지 경로와 라벨이 순서대로 저장되어 있음
            image_path = test_dataset.samples[sample_index][0]

            all_results.append({
                "index": sample_index,
                "image_path": image_path,
                "true_label_idx": true_label_idx,
                "pred_label_idx": pred_label_idx,
                "true_label": class_names[true_label_idx],
                "pred_label": class_names[pred_label_idx],
                "pred_prob": pred_prob,
                "is_correct": true_label_idx == pred_label_idx
            })

            sample_index += 1

results_df = pd.DataFrame(all_results)

print("Test 전체 이미지 수:", len(results_df))
print("정분류 개수:", results_df["is_correct"].sum())
print("오분류 개수:", (~results_df["is_correct"]).sum())

display(results_df.head())

In [ ]:
# =========================
# 6단계-5. 오분류 데이터 추출
# =========================

misclassified_df = results_df[results_df["is_correct"] == False].copy()

print("오분류 이미지 수:", len(misclassified_df))

# 예측 확률이 높은 오분류부터 확인
misclassified_df = misclassified_df.sort_values(by="pred_prob", ascending=False)

display(misclassified_df.head(20))

In [ ]:
# =========================
# 6단계-6. 오분류 결과 상세 출력
# =========================

print("=" * 80)
print("오분류 샘플 목록")
print("=" * 80)

for idx, row in misclassified_df.head(30).iterrows():
    print(f"[Index {row['index']}]")
    print(f"이미지 경로: {row['image_path']}")
    print(f"실제 라벨: {row['true_label']}")
    print(f"예측 라벨: {row['pred_label']}")
    print(f"예측 확률: {row['pred_prob']:.4f}")
    print("-" * 80)

In [ ]:
# =========================
# 6단계-7. Test 샘플 예측 결과 시각화
# =========================

def visualize_sample_predictions(results_df, save_path, num_samples=20):
    """
    Test 데이터 일부에 대해 실제 라벨, 예측 라벨, 예측 확률을 시각화
    정분류는 제목에 O, 오분류는 X로 표시
    """

    sample_df = results_df.sample(
        n=min(num_samples, len(results_df)),
        random_state=42
    )

    cols = 5
    rows = int(np.ceil(len(sample_df) / cols))

    plt.figure(figsize=(20, rows * 4))

    for plot_idx, (_, row) in enumerate(sample_df.iterrows()):
        image = Image.open(row["image_path"]).convert("RGB")

        mark = "O" if row["is_correct"] else "X"

        title = (
            f"{mark}\n"
            f"True: {row['true_label']}\n"
            f"Pred: {row['pred_label']}\n"
            f"Prob: {row['pred_prob']:.2f}"
        )

        plt.subplot(rows, cols, plot_idx + 1)
        plt.imshow(image)
        plt.title(title, fontsize=10)
        plt.axis("off")

    plt.suptitle(f"{analysis_model_name} Sample Predictions", fontsize=18)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("샘플 예측 이미지 저장 완료:", save_path)


sample_predictions_path = os.path.join(RESULT_DIR, "sample_predictions.png")

visualize_sample_predictions(
    results_df=results_df,
    save_path=sample_predictions_path,
    num_samples=20
)

In [ ]:
# =========================
# 6단계-8. 오분류 이미지 시각화
# =========================

def visualize_misclassified_samples(misclassified_df, save_path, num_samples=20):
    """
    오분류된 이미지 일부를 시각화
    실제 라벨, 예측 라벨, 예측 확률을 함께 출력
    """

    if len(misclassified_df) == 0:
        print("오분류된 이미지가 없습니다.")
        return

    sample_df = misclassified_df.head(num_samples)

    cols = 5
    rows = int(np.ceil(len(sample_df) / cols))

    plt.figure(figsize=(20, rows * 4))

    for plot_idx, (_, row) in enumerate(sample_df.iterrows()):
        image = Image.open(row["image_path"]).convert("RGB")

        title = (
            f"True: {row['true_label']}\n"
            f"Pred: {row['pred_label']}\n"
            f"Prob: {row['pred_prob']:.2f}"
        )

        plt.subplot(rows, cols, plot_idx + 1)
        plt.imshow(image)
        plt.title(title, fontsize=10)
        plt.axis("off")

    plt.suptitle(f"{analysis_model_name} Misclassified Samples", fontsize=18)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("오분류 이미지 저장 완료:", save_path)


misclassified_samples_path = os.path.join(RESULT_DIR, "misclassified_samples.png")

visualize_misclassified_samples(
    misclassified_df=misclassified_df,
    save_path=misclassified_samples_path,
    num_samples=20
)

In [ ]:
# =========================
# 6단계-9. 유사 클래스 혼동 분석 함수 정의
# =========================

def normalize_class_name(name):
    """
    클래스명 비교를 위해 공백을 제거하는 함수
    예: '연근 조림' → '연근조림'
    """
    return name.replace(" ", "")


def analyze_pair_confusion(results_df, class_a, class_b):
    """
    두 클래스 간 혼동 여부를 분석하는 함수

    class_a가 실제인데 class_b로 예측한 경우
    class_b가 실제인데 class_a로 예측한 경우
    를 각각 계산함
    """

    norm_a = normalize_class_name(class_a)
    norm_b = normalize_class_name(class_b)

    temp_df = results_df.copy()
    temp_df["true_label_norm"] = temp_df["true_label"].apply(normalize_class_name)
    temp_df["pred_label_norm"] = temp_df["pred_label"].apply(normalize_class_name)

    a_to_b = temp_df[
        (temp_df["true_label_norm"] == norm_a) &
        (temp_df["pred_label_norm"] == norm_b)
    ]

    b_to_a = temp_df[
        (temp_df["true_label_norm"] == norm_b) &
        (temp_df["pred_label_norm"] == norm_a)
    ]

    total_confusion = len(a_to_b) + len(b_to_a)

    return {
        "class_pair": f"{class_a} vs {class_b}",
        f"{class_a}_predicted_as_{class_b}": len(a_to_b),
        f"{class_b}_predicted_as_{class_a}": len(b_to_a),
        "total_confusion": total_confusion
    }

In [ ]:
# =========================
# 6단계-10. 유사 클래스 혼동 여부 확인
# =========================

similar_class_pairs = [
    ("더덕구이", "두부조림"),
    ("동그랑땡", "생선전"),
    ("파전", "계란후라이"),
    ("연근조림", "땅콩조림"),
    ("고사리나물", "미역줄기볶음")
]

pair_confusion_results = []

for class_a, class_b in similar_class_pairs:
    result = analyze_pair_confusion(
        results_df=results_df,
        class_a=class_a,
        class_b=class_b
    )
    pair_confusion_results.append(result)

pair_confusion_df = pd.DataFrame(pair_confusion_results)

display(pair_confusion_df)

In [ ]:
# =========================
# 6단계-11. 유사 클래스 혼동 결과 해석
# =========================

print("=" * 80)
print("유사 클래스 혼동 분석 결과")
print("=" * 80)

for _, row in pair_confusion_df.iterrows():
    print(f"\n[{row['class_pair']}]")

    confusion_cols = [
        col for col in pair_confusion_df.columns
        if col not in ["class_pair", "total_confusion"]
    ]

    for col in confusion_cols:
        if col in row:
            print(f"{col}: {row[col]}")

    print(f"총 혼동 수: {row['total_confusion']}")

    if row["total_confusion"] > 0:
        print("→ 해당 클래스 쌍 사이에서 혼동이 발생했습니다.")
    else:
        print("→ 해당 클래스 쌍 사이의 직접적인 혼동은 확인되지 않았습니다.")

In [ ]:
# =========================
# 6단계-12. 유사 클래스 혼동 샘플 확인
# =========================

def get_pair_confusion_samples(results_df, class_a, class_b):
    """
    특정 유사 클래스 쌍에서 실제-예측이 서로 뒤바뀐 오분류 샘플만 추출
    """

    norm_a = normalize_class_name(class_a)
    norm_b = normalize_class_name(class_b)

    temp_df = results_df.copy()
    temp_df["true_label_norm"] = temp_df["true_label"].apply(normalize_class_name)
    temp_df["pred_label_norm"] = temp_df["pred_label"].apply(normalize_class_name)

    pair_samples = temp_df[
        (
            (temp_df["true_label_norm"] == norm_a) &
            (temp_df["pred_label_norm"] == norm_b)
        )
        |
        (
            (temp_df["true_label_norm"] == norm_b) &
            (temp_df["pred_label_norm"] == norm_a)
        )
    ]

    return pair_samples


for class_a, class_b in similar_class_pairs:
    pair_samples = get_pair_confusion_samples(results_df, class_a, class_b)

    print("=" * 80)
    print(f"{class_a} vs {class_b} 혼동 샘플 수: {len(pair_samples)}")
    print("=" * 80)

    if len(pair_samples) > 0:
        display(pair_samples[[
            "index",
            "image_path",
            "true_label",
            "pred_label",
            "pred_prob"
        ]].head())
    else:
        print("해당 클래스 쌍의 직접적인 혼동 샘플이 없습니다.")

In [ ]:
# =========================
# 6단계-13. 저장 결과 확인
# =========================

expected_files = [
    "sample_predictions.png",
    "misclassified_samples.png"
]

print("results 폴더 저장 파일 확인")
print("=" * 50)

for file_name in expected_files:
    file_path = os.path.join(RESULT_DIR, file_name)
    print(f"{file_name}: {os.path.exists(file_path)}")